# MCP Servers: Role-Based Access Control (RBAC) Guide

**Purpose**: Examples and templates for creating and managing Snowflake-managed MCP servers with proper RBAC.

**Target Audience**: Snowflake Administrators, Data Engineers, AI/ML Engineers

**Last Updated**: March 2026

**Version**: 2.0

## Overview

Snowflake-managed MCP servers let AI agents securely interact with Snowflake data via the Model Context Protocol standard. This runbook covers creating MCP servers with RBAC, integrating Cortex tools (Analyst, Search, Agents), configuring custom tools, and managing operations.

### Prerequisites

- `ACCOUNTADMIN` role
- Existing Cortex Agent (from Notebook 04), Search service (02), and Semantic View (03)
- Warehouse for query execution

**Note**: Snowflake supports Model Context Protocol revision 2025-06-18.

### Key Concepts

**MCP Server Privileges**: `CREATE MCP SERVER`, `USAGE`, `OWNERSHIP`

**Tool Types**: `CORTEX_ANALYST_MESSAGE`, `CORTEX_SEARCH_SERVICE_QUERY`, `CORTEX_AGENT_RUN`, `SYSTEM_EXECUTE_SQL`, `GENERIC`

**Authentication**: OAuth (recommended) or Programmatic Access Token (PAT)

---

## Part 1: Configuration Variables

Set these variables to match your environment. All object names will be derived from these base values.

In [ ]:
USE SECONDARY ROLES NONE;

-- ============================================================================
-- CONFIGURATION VARIABLES
-- ============================================================================

-- Database and Schema
SET DATABASE_NAME = 'RUNBOOKS_DB';
SET SCHEMA_NAME = 'MCP';

-- Full qualified paths (required for IDENTIFIER() function)
SET FULL_SCHEMA_PATH = 'RUNBOOKS_DB.MCP';

-- Role Configuration
SET MCP_ADMIN_ROLE = 'MCP_ADMIN_ROLE';  -- Role for creating and managing MCP servers
SET MCP_USER_ROLE = 'MCP_USER_ROLE';    -- Role for connecting to MCP servers

-- Warehouse
SET WAREHOUSE_NAME = 'COMPUTE_WH';

-- MCP Server Name
SET MCP_SERVER_NAME = 'ANALYTICS_MCP_SERVER';

-- ============================================================================
-- PATHS TO EXISTING TOOLS FROM PREVIOUS NOTEBOOKS
-- ============================================================================

-- Cortex Search Service (from 02_CortexSearch notebook)
SET SEARCH_SERVICE_PATH = 'RUNBOOKS_DB.CORTEX_SEARCH.PRODUCT_SEARCH_SERVICE';

-- Semantic View (from 03_CortexAnalyst notebook)
SET SEMANTIC_VIEW_PATH = 'RUNBOOKS_DB.CORTEX_ANALYST.SALES_SEMANTIC_VIEW_RESTRICTED';

-- Cortex Agent (from 04_CortexAgents notebook)
SET AGENT_PATH = 'RUNBOOKS_DB.CORTEX_AGENTS.SALES_ANALYTICS_AGENT';

-- ============================================================================

-- Display configuration
SELECT 
    $DATABASE_NAME AS DATABASE_NAME,
    $SCHEMA_NAME AS SCHEMA_NAME,
    $FULL_SCHEMA_PATH AS FULL_SCHEMA_PATH,
    $MCP_ADMIN_ROLE AS MCP_ADMIN_ROLE,
    $MCP_USER_ROLE AS MCP_USER_ROLE,
    $WAREHOUSE_NAME AS WAREHOUSE_NAME,
    $MCP_SERVER_NAME AS MCP_SERVER_NAME,
    $SEARCH_SERVICE_PATH AS SEARCH_SERVICE_PATH,
    $SEMANTIC_VIEW_PATH AS SEMANTIC_VIEW_PATH,
    $AGENT_PATH AS AGENT_PATH;

---

## Part 2: RBAC Setup

Create custom roles and establish privilege hierarchy for MCP server management.

### 2.1: Create Infrastructure

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Create database if not exists (likely already exists from previous notebooks)
CREATE DATABASE IF NOT EXISTS IDENTIFIER($DATABASE_NAME);

-- Create schema for MCP servers
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($FULL_SCHEMA_PATH);

-- Create warehouse for MCP operations
CREATE WAREHOUSE IF NOT EXISTS IDENTIFIER($WAREHOUSE_NAME)
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

### 2.2: Create Custom Roles

We'll create a two-role hierarchy for MCP servers:

1. **MCP Admin Role**: Can create and manage MCP servers
2. **MCP User Role**: Can connect to MCP servers and discover tools

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Create custom roles
CREATE ROLE IF NOT EXISTS IDENTIFIER($MCP_ADMIN_ROLE)
    COMMENT = 'Role for creating and managing MCP servers';

CREATE ROLE IF NOT EXISTS IDENTIFIER($MCP_USER_ROLE)
    COMMENT = 'Role for connecting to MCP servers and discovering tools';

-- Set up role hierarchy
GRANT ROLE IDENTIFIER($MCP_USER_ROLE) TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);
GRANT ROLE IDENTIFIER($MCP_ADMIN_ROLE) TO ROLE SYSADMIN;

### 2.3: Grant Infrastructure Privileges

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Grant database and schema privileges to MCP admin role
GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);
GRANT CREATE MCP SERVER ON SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);

-- Grant database and schema privileges to MCP user role
GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($MCP_USER_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($FULL_SCHEMA_PATH) TO ROLE IDENTIFIER($MCP_USER_ROLE);

-- Grant warehouse usage to both roles
GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);
GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($MCP_USER_ROLE);

---

## Part 3: Grant Access to Existing Tools

Before creating the MCP server, we need to ensure the MCP admin role has access to the tools that will be exposed through the MCP server.

### 3.1: Grant Access to Cortex Search Service

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Grant database and schema access for the search service
GRANT USAGE ON DATABASE RUNBOOKS_DB TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);
GRANT USAGE ON SCHEMA RUNBOOKS_DB.CORTEX_SEARCH TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);

-- Grant USAGE on the Cortex Search Service
GRANT USAGE ON CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_PATH) 
    TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);

-- Also grant to MCP user role (they will need it to use the tool through MCP)
GRANT USAGE ON DATABASE RUNBOOKS_DB TO ROLE IDENTIFIER($MCP_USER_ROLE);
GRANT USAGE ON SCHEMA RUNBOOKS_DB.CORTEX_SEARCH TO ROLE IDENTIFIER($MCP_USER_ROLE);
GRANT USAGE ON CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_PATH) 
    TO ROLE IDENTIFIER($MCP_USER_ROLE);

### 3.2: Grant Access to Semantic View

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Grant database and schema access for the semantic view
GRANT USAGE ON SCHEMA RUNBOOKS_DB.CORTEX_ANALYST TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);

-- Grant SELECT and REFERENCES on the Semantic View
GRANT SELECT ON SEMANTIC VIEW IDENTIFIER($SEMANTIC_VIEW_PATH) 
    TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);
GRANT REFERENCES ON SEMANTIC VIEW IDENTIFIER($SEMANTIC_VIEW_PATH) 
    TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);

-- Also grant to MCP user role
GRANT USAGE ON SCHEMA RUNBOOKS_DB.CORTEX_ANALYST TO ROLE IDENTIFIER($MCP_USER_ROLE);
GRANT SELECT ON SEMANTIC VIEW IDENTIFIER($SEMANTIC_VIEW_PATH) 
    TO ROLE IDENTIFIER($MCP_USER_ROLE);
GRANT REFERENCES ON SEMANTIC VIEW IDENTIFIER($SEMANTIC_VIEW_PATH) 
    TO ROLE IDENTIFIER($MCP_USER_ROLE);
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE IDENTIFIER($MCP_USER_ROLE); -- to invoke tool
GRANT SELECT ON TABLE RUNBOOKS_DB.CORTEX_ANALYST.SALES_DATA_RESTRICTED TO ROLE IDENTIFIER($MCP_USER_ROLE); -- To query underlying table with agent or semantic view --> Row access policy is being enforced 

### 3.3: Grant Access to Cortex Agent

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Grant database and schema access for the agent
GRANT USAGE ON SCHEMA RUNBOOKS_DB.CORTEX_AGENTS TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);

-- Grant USAGE on the Agent
GRANT USAGE ON AGENT IDENTIFIER($AGENT_PATH) 
    TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);

-- Also grant to MCP user role
GRANT USAGE ON SCHEMA RUNBOOKS_DB.CORTEX_AGENTS TO ROLE IDENTIFIER($MCP_USER_ROLE);
GRANT USAGE ON AGENT IDENTIFIER($AGENT_PATH) TO ROLE IDENTIFIER($MCP_USER_ROLE);

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE IDENTIFIER($MCP_USER_ROLE); -- To invoke through SI
GRANT SELECT, REFERENCES ON SEMANTIC VIEW RUNBOOKS_DB.CORTEX_ANALYST.SALES_SEMANTIC_VIEW_RESTRICTED TO ROLE IDENTIFIER($MCP_USER_ROLE);

---

## Part 4: Create Custom Tools (Optional)

Before creating the MCP server, let's create some custom tools (UDFs and stored procedures) that can be exposed through the MCP server.

### 4.1: Create Sample UDFs

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);

-- Create a simple UDF for calculations
CREATE OR REPLACE FUNCTION CALCULATE_DISCOUNT(price FLOAT, discount_percent FLOAT)
RETURNS FLOAT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.9'
HANDLER = 'calculate_discount'
AS $$
def calculate_discount(price: float, discount_percent: float) -> float:
    """
    Calculates the final price after applying a discount percentage.
    """
    discount_amount = price * (discount_percent / 100)
    return price - discount_amount
$$;

-- Create a UDF that returns structured data
CREATE OR REPLACE FUNCTION GET_SALES_SUMMARY(region STRING)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.9'
HANDLER = 'get_sales_summary'
AS $$
def get_sales_summary(region: str) -> dict:
    """
    Returns a summary of sales data for a given region.
    In production, this would query actual data.
    """
    return {
        "region": region,
        "total_sales": 150000,
        "num_transactions": 1200,
        "avg_sale": 125
    }
$$;

-- Test the UDFs
SELECT CALCULATE_DISCOUNT(100, 20) AS discounted_price;
SELECT GET_SALES_SUMMARY('West') AS sales_summary;

### 4.2: Grant Access to Custom Tools

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Grant USAGE on UDFs to MCP roles
GRANT USAGE ON FUNCTION CALCULATE_DISCOUNT(FLOAT, FLOAT) TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);
GRANT USAGE ON FUNCTION CALCULATE_DISCOUNT(FLOAT, FLOAT) TO ROLE IDENTIFIER($MCP_USER_ROLE);

GRANT USAGE ON FUNCTION GET_SALES_SUMMARY(STRING) TO ROLE IDENTIFIER($MCP_ADMIN_ROLE);
GRANT USAGE ON FUNCTION GET_SALES_SUMMARY(STRING) TO ROLE IDENTIFIER($MCP_USER_ROLE);

---

## Part 5: Create MCP Server

Now we'll create the MCP server that exposes all the tools we've configured.

### 5.1: Understanding MCP Server Specification

Each tool in the spec has: `name`, `type`, `identifier` (fully-qualified Snowflake object), `description`, `title`, and `config` (warehouse, input schema, etc.).

### 5.2: Create the MCP Server

In [ ]:
USE ROLE IDENTIFIER($MCP_ADMIN_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);

CREATE OR REPLACE MCP SERVER IDENTIFIER($MCP_SERVER_NAME)
FROM SPECIFICATION $$
tools:
  # Cortex Analyst Tool - Query sales data using semantic view
  - name: "sales-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "RUNBOOKS_DB.CORTEX_ANALYST.SALES_SEMANTIC_VIEW_RESTRICTED"
    description: "Semantic view for querying sales data including revenue, products, and regional performance"
    title: "Sales Data Analyst"
  
  # Cortex Search Tool - Search product documentation
  - name: "product-search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "RUNBOOKS_DB.CORTEX_SEARCH.PRODUCT_SEARCH_SERVICE"
    description: "Search service for product documentation, specifications, and support information"
    title: "Product Documentation Search"
  
  # Cortex Agent Tool - Orchestrated analytics agent
  - name: "analytics-agent"
    type: "CORTEX_AGENT_RUN"
    identifier: "RUNBOOKS_DB.CORTEX_AGENTS.SALES_ANALYTICS_AGENT"
    description: "AI agent that orchestrates sales analysis and product search capabilities"
    title: "Sales Analytics Agent"
  
  # SQL Execution Tool - Execute arbitrary SQL queries
  - name: "sql-execution"
    type: "SYSTEM_EXECUTE_SQL"
    description: "Execute SQL queries against the connected Snowflake database"
    title: "SQL Query Executor"
    config:
        warehouse: "COMPUTE_WH"
  
  # Custom Tool 1 - Calculate discounts
  - name: "calculate-discount"
    type: "GENERIC"
    identifier: "RUNBOOKS_DB.MCP.CALCULATE_DISCOUNT"
    description: "Calculate the final price after applying a discount percentage"
    title: "Discount Calculator"
    config:
      type: "function"
      warehouse: "COMPUTE_WH"
      input_schema:
        type: "object"
        properties:
          price:
            description: "Original price"
            type: "number"
          discount_percent:
            description: "Discount percentage (e.g., 20 for 20%)"
            type: "number"
  
  # Custom Tool 2 - Get sales summary
  - name: "sales-summary"
    type: "GENERIC"
    identifier: "RUNBOOKS_DB.MCP.GET_SALES_SUMMARY"
    description: "Get a summary of sales data for a specific region"
    title: "Regional Sales Summary"
    config:
      type: "function"
      warehouse: "COMPUTE_WH"
      input_schema:
        type: "object"
        properties:
          region:
            description: "Region name (e.g., West, East, Central)"
            type: "string"
$$;

### 5.3: Verify MCP Server Creation

In [ ]:
-- Show all MCP servers in the schema
SHOW MCP SERVERS IN SCHEMA IDENTIFIER($FULL_SCHEMA_PATH);

-- Describe the MCP server to see its configuration
DESCRIBE MCP SERVER IDENTIFIER($MCP_SERVER_NAME);

---

## Part 6: Grant MCP Server Access

Grant USAGE privilege on the MCP server to allow users to connect and discover tools.

### 6.1: Grant USAGE Privilege

In [ ]:
USE ROLE IDENTIFIER($MCP_ADMIN_ROLE);

-- Grant USAGE privilege on the MCP server to the user role
GRANT USAGE ON MCP SERVER IDENTIFIER($MCP_SERVER_NAME) 
    TO ROLE IDENTIFIER($MCP_USER_ROLE); -- Need to add additional permissions on agent and tools.

-- Note: Users with USAGE privilege can discover and invoke tools,
-- but they still need appropriate privileges on the underlying tools themselves

### 6.2: Verify Grants

In [ ]:
-- View grants on the MCP server
SHOW GRANTS ON MCP SERVER IDENTIFIER($MCP_SERVER_NAME);

-- View all grants to the MCP user role
SHOW GRANTS TO ROLE IDENTIFIER($MCP_USER_ROLE);

---

## Part 7: MCP Server Connection and Usage

Understanding how to connect to and use the MCP server.

### 7.1: Connection Configuration

MCP clients connect using standard Snowflake connection parameters plus the MCP server identifier. Authentication via OAuth or PAT.

### 7.2: Tool Discovery

Once connected, MCP clients discover available tools through the standard MCP protocol.

### 7.3: Get MCP Server Connection Details

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Get the account locator for connection string
SELECT CURRENT_ACCOUNT() AS ACCOUNT_LOCATOR,
       CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() AS ACCOUNT_IDENTIFIER,
       $FULL_SCHEMA_PATH || '.' || $MCP_SERVER_NAME AS MCP_SERVER_PATH,
       'https://' || CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() || 
       '.snowflakecomputing.com/api/v2/mcp/servers/' || 
       $FULL_SCHEMA_PATH || '.' || $MCP_SERVER_NAME AS MCP_SERVER_URL;

---

## Part 8: Managing MCP Servers

Common operations for managing MCP servers.

### 8.1: Modify MCP Server Configuration

To modify an MCP server, use `CREATE OR REPLACE` with the updated specification.

In [ ]:
USE ROLE IDENTIFIER($MCP_ADMIN_ROLE);

-- Example: Add a new tool to the MCP server
-- (Uncomment and modify as needed)

/*
CREATE OR REPLACE MCP SERVER IDENTIFIER($MCP_SERVER_NAME)
FROM SPECIFICATION $$
tools:
  # ... existing tools ...
  
  # New tool
  - name: "new-tool"
    type: "GENERIC"
    identifier: "RUNBOOKS_DB.MCP.NEW_FUNCTION"
    description: "Description of new tool"
    title: "New Tool"
    config:
      type: "function"
      warehouse: "COMPUTE_WH"
      input_schema:
        type: "object"
        properties:
          param1:
            description: "Parameter description"
            type: "string"
$$;
*/

### 8.2: Transfer Ownership

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Transfer ownership to a different role (if needed)
-- GRANT OWNERSHIP ON MCP SERVER IDENTIFIER($MCP_SERVER_NAME) TO ROLE <new_owner_role>;

# 8.3: Drop MCP Server

In [ ]:
USE ROLE IDENTIFIER($MCP_ADMIN_ROLE);

-- Drop the MCP server (commented out for safety)
-- DROP MCP SERVER IF EXISTS IDENTIFIER($MCP_SERVER_NAME);

---

## Part 8: Cleanup (Optional)

Run these commands to clean up all resources created in this runbook.

**Note**: This cleanup ONLY removes objects created in this notebook. It does NOT remove tools from previous notebooks.

**Warning**: Schema and database drops are commented out to prevent accidental deletion.

In [ ]:
--USE ROLE ACCOUNTADMIN;

-- Drop the MCP server
--DROP MCP SERVER IF EXISTS IDENTIFIER($MCP_SERVER_NAME);

-- Drop custom UDFs
--DROP FUNCTION IF EXISTS CALCULATE_DISCOUNT(FLOAT, FLOAT);
--DROP FUNCTION IF EXISTS GET_SALES_SUMMARY(STRING);

-- Drop roles
--DROP ROLE IF EXISTS IDENTIFIER($MCP_ADMIN_ROLE);
--DROP ROLE IF EXISTS IDENTIFIER($MCP_USER_ROLE);

-- Drop warehouse (commented out as it may be shared)
-- DROP WAREHOUSE IF EXISTS IDENTIFIER($WAREHOUSE_NAME);

-- Drop schema and database (commented out for safety)
-- DROP SCHEMA IF EXISTS IDENTIFIER($FULL_SCHEMA_PATH);
-- DROP DATABASE IF EXISTS IDENTIFIER($DATABASE_NAME);

-- NOTE: We DO NOT drop the Cortex Search service, Semantic View, or Agent
-- as they belong to previous notebooks and may still be in use

---

## Additional Resources

- [Snowflake-managed MCP Server Overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-mcp)
- [Getting Started with Managed Snowflake MCP Server](https://www.snowflake.com/en/developers/guides/getting-started-with-snowflake-mcp-server/)
- [Cortex Agents Documentation](https://docs.snowflake.com/user-guide/snowflake-cortex/cortex-agents)

---

**Version:** 2.0  
**Last Updated:** March 2026